# **Packages and Real-World OOP Design**

**DS212 — Object-Oriented Programming**

**Week 8:** · **Suggested class time: 5 hours (2-hour lecture + 3-hour laboratory)**

**ILO :** Organize object-oriented code into packages and relate OOP principles to real-world applications.

So far, most of our classes have lived inside a single notebook. Real applications contain **dozens or hundreds of classes**, written by several people, and maintained for years. Two questions become important:

1. **Where should each class live?** → *modules, packages, and namespaces*
2. **How should classes divide the work between them?** → *class collaboration and responsibility*

```{mermaid}
flowchart LR
    A["One notebook<br/>everything together"] --> B["Modules<br/>related code per file"]
    B --> C["Packages<br/>related modules per folder"]
    C --> D["Application<br/>collaborating classes"]
```

This lesson connects both questions to **real-world object-oriented modeling**.

## **Modules**

A **module** is simply a `.py` file. Its name is the file name without `.py`. Everything defined at the top level of the file — functions, classes, variables — becomes available to code that **imports** it.

```text
grading.py
├── PASSING_GRADE = 75          (variable)
├── def letter_grade(score)     (function)
└── class Student               (class)
```

```python
# main.py
import grading

student = grading.Student("Ana")
print(grading.letter_grade(92))
```

When Python imports a module, it:

1. finds the file (searching the folders listed in `sys.path`);
2. **runs** the file once, top to bottom;
3. stores the result as a **module object**, so later imports reuse it.

### **`if __name__ == "__main__":`**

Because importing *runs* a module, code meant only for testing or demonstration should be guarded:

```python
# grading.py
def letter_grade(score):
    ...

if __name__ == "__main__":
    # runs only with: python grading.py
    # does NOT run on: import grading
    print(letter_grade(92))
```

`__name__` is `"__main__"` when the file is run directly, and the module's own name when it is imported.

## **Packages**

A **package** is a **folder of modules**. A folder becomes a regular package when it contains an `__init__.py` file.

```text
campus/                     ← package
├── __init__.py             ← marks the folder as a package; runs on import
├── people.py               ← module: Person, Student, Instructor
├── academics.py            ← module: Course, Section
└── services/               ← sub-package
    ├── __init__.py
    └── enrollment.py       ← module: EnrollmentService
```

Imports follow the folder structure using dots:

```python
from campus.people import Student
from campus.services.enrollment import EnrollmentService
```

Inside a package, modules can refer to each other with **relative imports**:

```python
# campus/services/enrollment.py
from ..people import Student          # go up one package, then into people
from ..academics import Course
```

### **The role of `__init__.py`**

`__init__.py` can be empty, but it is often used to define the package's **public interface** — the names users should import:

```python
# campus/__init__.py
from .people import Student, Instructor
from .academics import Course

__all__ = ["Student", "Instructor", "Course"]
__version__ = "0.1.0"
```

Users can then write the shorter:

```python
from campus import Student, Course
```

This is **abstraction at the package level**: users see a small, stable interface, while the internal file layout can change freely.

### **Building a package in the notebook**

The cell below creates a small `campus` package on disk (in a temporary folder), then imports it — exactly as if you had created the files in your editor.

In [ ]:
import sys
import tempfile
import textwrap
from pathlib import Path

root = Path(tempfile.mkdtemp())
pkg = root / "campus"
(pkg / "services").mkdir(parents=True)

files = {
    "__init__.py": '''
        from .people import Student
        from .academics import Course

        __all__ = ["Student", "Course"]
        __version__ = "0.1.0"
    ''',
    "people.py": '''
        class Student:
            def __init__(self, name, student_id):
                self.name = name
                self.student_id = student_id

            def __repr__(self):
                return f"Student({self.name!r})"
    ''',
    "academics.py": '''
        class Course:
            def __init__(self, code, capacity):
                self.code = code
                self.capacity = capacity
                self.roster = []

            def has_slot(self):
                return len(self.roster) < self.capacity
    ''',
    "services/__init__.py": "",
    "services/enrollment.py": '''
        from ..academics import Course
        from ..people import Student


        class EnrollmentService:
            def enroll(self, student: Student, course: Course):
                if not course.has_slot():
                    return f"{course.code} is full"
                course.roster.append(student)
                return f"{student.name} enrolled in {course.code}"
    ''',
}
for name, source in files.items():
    (pkg / name).write_text(textwrap.dedent(source).lstrip(), encoding="utf-8")

for p in sorted(root.rglob("*.py")):
    print(p.relative_to(root))

In [ ]:
sys.path.insert(0, str(root))          # let Python find the package

import campus
from campus import Student, Course
from campus.services.enrollment import EnrollmentService

service = EnrollmentService()
ds212 = Course("DS212", capacity=2)

for s in [Student("Ana", "001"), Student("Ben", "002"), Student("Carlo", "003")]:
    print(service.enroll(s, ds212))

print(ds212.roster, campus.__version__)

## **Namespaces**

A **namespace** is a mapping from **names to objects**. Every module, class, function call, and object has its own namespace, which is why two modules can both define a function called `load()` without conflict.

```{mermaid}
flowchart TB
    B["Built-in namespace<br/>print, len, int"] --> G["Module (global) namespace<br/>campus.people: Student"]
    G --> E["Enclosing function namespace"]
    E --> L["Local namespace<br/>variables inside a function call"]
```

Python looks up a name in the order **L**ocal → **E**nclosing → **G**lobal → **B**uilt-in (the **LEGB** rule).

Modules **are** namespaces, and dot notation looks names up inside them:

In [ ]:
import math
import statistics

data = [2, 4, 4, 4, 5, 5, 7, 9]

# two modules, each with its own name for related ideas
print(math.sqrt(16), statistics.pstdev(data))

# a module's namespace can be inspected like any object's
print([name for name in dir(statistics) if name.startswith("m")])
print(type(math), math.__name__)

### **Import styles and their trade-offs**

| Style                                   | Example                             | Trade-off                                           |
| --------------------------------------- | ----------------------------------- | --------------------------------------------------- |
| Import the module                       | `import statistics`                 | clearest origin: `statistics.mean()`                |
| Import with alias                       | `import pandas as pd`               | short and conventional for well-known libraries    |
| Import specific names                   | `from campus import Student`        | concise; fine for a few names                       |
| Wildcard import                         | `from campus import *`              | **avoid** — hides where names came from and may overwrite your own names |

> **Tip:** never name your own file after a standard library module (`random.py`, `statistics.py`, `csv.py`). Your file will be imported instead of the real module, causing confusing errors.

## **Class Collaboration and Responsibility**

Splitting code into packages answers *where* classes live. A harder question is *what each class should do*. A well-designed object-oriented system is a group of **collaborating objects**, each with a clear **responsibility**.

### **Responsibilities**

A responsibility is something a class **knows** or **does**:

* **knowing** — `Course` knows its code, capacity, and roster;
* **doing** — `Course` can tell whether it has an available slot.

### **CRC cards**

**Class–Responsibility–Collaborator (CRC) cards**, introduced by Kent Beck and Ward Cunningham, are a simple, low-tech way to design before coding. Each card lists one class, what it is responsible for, and which other classes it needs.

```text
┌─────────────────────────────────────────────────────┐
│ Course                                              │
├──────────────────────────────┬──────────────────────┤
│ Responsibilities             │ Collaborators        │
├──────────────────────────────┼──────────────────────┤
│ know code, title, capacity   │                      │
│ keep the roster              │ Student              │
│ report whether it has slots  │                      │
│ compute the class average    │ Student, Grade       │
└──────────────────────────────┴──────────────────────┘
```

Teams often write CRC cards on index cards and **role-play scenarios**: *"A student asks to enroll. Who receives the request? Who checks capacity? Who records the enrollment?"* Gaps and overloaded classes become obvious quickly.

### **Guidelines for assigning responsibilities**

| Guideline                  | Question to ask                                              | Example                                                     |
| -------------------------- | ------------------------------------------------------------ | ----------------------------------------------------------- |
| **Information expert**     | Which class already has the data needed?                     | `Course` computes its own average because it holds the roster |
| **Creator**                | Which class contains or closely uses the new object?         | `Course` creates its own `Assignment` objects               |
| **High cohesion**          | Does everything in this class belong together?               | `ReportExporter` should not also send emails                |
| **Low coupling**           | Can this class work while knowing as few others as possible? | `EnrollmentService` depends on `Course.has_slot()`, not on the roster list |
| **Tell, don't ask**        | Am I pulling data out to make a decision the object could make itself? | `course.has_slot()` instead of `len(course.roster) < course.capacity` |

### **Visualizing collaboration**

A **sequence diagram** shows how objects send messages to one another over time:

```{mermaid}
sequenceDiagram
    participant UI as Registrar UI
    participant ES as EnrollmentService
    participant C as Course
    participant S as Student
    UI->>ES: enroll(student, course)
    ES->>C: has_slot()
    C-->>ES: True
    ES->>C: add(student)
    ES->>S: add_course(course)
    ES-->>UI: "Ana enrolled in DS212"
```

Class diagrams show **structure**; sequence diagrams show **behavior**. Together they describe how responsibilities are shared.

## **Real-World Object-Oriented Modeling**

Let us model a real problem from start to finish.

### **1 — Requirements**

> The **campus library** lends **books** to **members**. A member may borrow at most **3 books** at a time. Each **loan** lasts **14 days**. When a book is returned late, the member is charged a **fine of ₱10 per day**. Staff must be able to see which books are currently **on loan**.

### **2 — Find candidate classes and responsibilities**

Nouns suggest classes and attributes; verbs suggest methods.

| Noun / verb      | Decision                                 |
| ---------------- | ---------------------------------------- |
| library          | class `Library` — coordinates lending    |
| book             | class `Book`                             |
| member           | class `Member`                           |
| loan             | class `Loan` — connects a member and a book over a time period |
| fine             | computed by `Loan` — *not* a separate class |
| 3 books, 14 days, ₱10 | constants / class attributes         |
| lend, return     | methods of `Library`                     |

### **3 — Design**

```{mermaid}
classDiagram
class Library {
    -dict books
    -list loans
    +add_book(book)
    +lend(book_id, member, day) Loan
    +return_book(book_id, day) int
    +on_loan() list
}
class Book {
    +String book_id
    +String title
    +bool available
}
class Member {
    +int MAX_LOANS$
    +String name
    -list active_loans
    +can_borrow() bool
}
class Loan {
    +int DAYS$
    +int FINE_PER_DAY$
    +Book book
    +Member member
    +date due
    +fine(returned) int
}
Library o-- Book : catalog
Library *-- Loan : records
Loan --> Book
Loan --> Member
```

Notice the responsibilities: `Member` knows its own **borrowing limit**, `Loan` knows how to compute its own **fine** (information expert), and `Library` **coordinates** the process without doing everyone else's work.

### **4 — Implementation**

In [ ]:
from datetime import date, timedelta


class Book:
    def __init__(self, book_id, title):
        self.book_id = book_id
        self.title = title
        self.available = True


class Member:
    MAX_LOANS = 3

    def __init__(self, name):
        self.name = name
        self.active_loans = []

    def can_borrow(self):
        return len(self.active_loans) < self.MAX_LOANS


class Loan:
    DAYS = 14
    FINE_PER_DAY = 10

    def __init__(self, book, member, start):
        self.book = book
        self.member = member
        self.due = start + timedelta(days=self.DAYS)

    def fine(self, returned):
        days_late = (returned - self.due).days
        return max(0, days_late) * self.FINE_PER_DAY


class Library:
    def __init__(self):
        self._books = {}
        self._loans = []

    def add_book(self, book):
        self._books[book.book_id] = book

    def lend(self, book_id, member, day):
        book = self._books[book_id]
        if not book.available:
            return None
        if not member.can_borrow():          # tell, don't ask
            return None
        loan = Loan(book, member, day)       # Library creates the Loan it records
        book.available = False
        member.active_loans.append(loan)
        self._loans.append(loan)
        return loan

    def return_book(self, book_id, day):
        loan = next(l for l in self._loans if l.book.book_id == book_id)
        self._loans.remove(loan)
        loan.member.active_loans.remove(loan)
        loan.book.available = True
        return loan.fine(day)

    def on_loan(self):
        return [(l.book.title, l.member.name, l.due.isoformat()) for l in self._loans]

In [ ]:
library = Library()
for book_id, title in [("B1", "Think Python"), ("B2", "Fluent Python"), ("B3", "Python for Data Analysis")]:
    library.add_book(Book(book_id, title))

ana = Member("Ana")
start = date(2026, 9, 1)

library.lend("B1", ana, start)
library.lend("B2", ana, start)
print("On loan:", library.on_loan())

fine = library.return_book("B1", date(2026, 9, 20))     # 5 days late
print("Fine for B1: ₱", fine)
print("On loan:", library.on_loan())

### **5 — Organize into a package**

Once the design works, the classes move from the notebook into a package:

```text
library_system/
├── __init__.py          from .library import Library ...
├── catalog.py           Book
├── members.py           Member
├── loans.py             Loan
└── library.py           Library
```

The same thinking — **requirements → classes and responsibilities → diagram → code → package** — is what you will use for your final OOP project.

## **Common Design Mistakes**

**The god class.** One class (`LibrarySystem`) does everything: stores books, validates members, computes fines, prints reports. Split responsibilities among cohesive classes.

**Anemic objects.** Classes only hold data while all logic lives in outside functions (`compute_fine(loan)`, `can_borrow(member)`). Move behavior to the class that owns the data.

**Circular imports.** `members.py` imports `loans.py`, which imports `members.py`. Usually a sign that responsibilities are tangled; move the shared piece to a third module or import inside the function that needs it.

**Mirroring the folder structure of a tutorial instead of your domain.** Name modules after concepts in *your* problem (`loans.py`), not after technical categories (`utils.py`, `helpers2.py`).

## **Summary**

| Concept              | Meaning                                                                   |
| -------------------- | ------------------------------------------------------------------------- |
| **Module**           | a `.py` file; a namespace of functions, classes, and variables            |
| **Package**          | a folder of modules with `__init__.py`; organizes related modules         |
| **Namespace**        | a mapping of names to objects; prevents name collisions                   |
| **Responsibility**   | something a class knows or does                                           |
| **Collaboration**    | objects sending messages (method calls) to accomplish a task              |
| **CRC card**         | a lightweight design tool: Class, Responsibilities, Collaborators         |

## **References**

Beck, K., & Cunningham, W. (1989). A laboratory for teaching object-oriented thinking. *ACM SIGPLAN Notices, 24*(10), 1–6. https://doi.org/10.1145/74878.74879

Fowler, M. (2003). *UML distilled: A brief guide to the standard object modeling language* (3rd ed.). Addison-Wesley.

Liang, Y. D. (2013). *Introduction to programming using Python*. Pearson.

Lott, S. F., & Phillips, D. (2021). *Python object-oriented programming* (4th ed.). Packt Publishing.

Martin, R. C. (2008). *Clean code: A handbook of agile software craftsmanship*. Prentice Hall.

Python Software Foundation. (2026). *The Python tutorial*. https://docs.python.org/3/tutorial/

Python Software Foundation. (2026). *The Python tutorial: 6. Modules*. https://docs.python.org/3/tutorial/modules.html

van Rossum, G., Warsaw, B., & Coghlan, A. (2001). *PEP 8 — Style guide for Python code*. Python Software Foundation. https://peps.python.org/pep-0008/